# 04 — From predictions to portfolio accounting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/a-quant-journey/blob/main/notebooks/teaching/04_predictions_to_portfolio_accounting.ipynb)

We translate the models’ predictions into hypothetical SPY positions and examine how trading timing and transaction costs affect portfolio value. The comparison uses historical prices and the predictions saved in Notebook 03.

**Google Colab:** Use the 2026.07 CPU runtime. On first execution, dependency installation may require a session restart. If prompted, restart the session and rerun the setup cell before continuing.

In [ ]:
# Colab only. Select a Python 3.12 CPU runtime.
import os

if (type(get_ipython()).__module__.startswith('google.colab.')
        and not os.environ.get('AQJ_COLAB_PREREQUISITE')):
    import runpy
    import subprocess
    from pathlib import Path

    branch = 'main'
    checkout = Path('/content/aqj-teaching') / branch
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--branch', branch, '--single-branch',
                        'https://github.com/FranQuant/a-quant-journey.git', str(checkout)], check=True)
    subprocess.run(['git', 'diff', '--quiet', 'HEAD'], cwd=checkout, check=True)
    runpy.run_path(str(checkout / 'scripts/colab_teaching.py'))['initialize'](4)

## 1. Reconstruct the validation sample

In Google Colab, the setup cell automatically prepares the dated predictions and manifest from Notebook 03. For local execution, run Notebook 03 first to generate those files. Use the same price cutoff (**2021-12-28**) and label-availability rules to reconstruct the rows those predictions must match.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/quant_journey').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start within the teaching checkout or runner snapshot.')
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/raw/tpq_eod_2014_2023.csv'
OUT = ROOT / 'outputs/track_e_multifamily'
OUT.mkdir(parents=True, exist_ok=True)
from src.quant_journey.execution.prediction_handoff import CUTOFF

from src.quant_journey.execution.prediction_handoff import (
    BENCHMARK, INITIAL_CASH, RESERVED_START, SCENARIOS, VALIDATION_START, read_predictions,
)
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.quant_journey.data.tpq import load_spy_prices_through
from src.quant_journey.execution.dataset import make_execution_dataset
from src.quant_journey.validation.label_aware import split_train_validation_by_dates
from src.quant_journey.execution.signals import map_signals_to_executions
from src.quant_journey.execution.positions import make_execution_positions
from src.quant_journey.backtesting.ledger import run_long_cash_ledger

### Load bounded prices and reconstruct the partitions

In [ ]:
prices = load_spy_prices_through(DATA, CUTOFF)
data = make_execution_dataset(prices)
train, validation = split_train_validation_by_dates(
    data, VALIDATION_START, RESERVED_START
)

### Validate the dated prediction handoff

In [ ]:
preds = read_predictions(OUT, train, validation)

### Build the accounting calendar

In [ ]:
# Keep the common close calendar through the last target's observed close.
calendar = prices.loc[validation.index.min():validation.label_available_at.max()]
print(
    'Validation feature dates:', preds.index.min().date(), 'to',
    preds.index.max().date(), 'features:', len(preds),
)
print(
    'Portfolio price calendar:', calendar.index.min().date(), 'to',
    calendar.index.max().date(), 'observations:', len(calendar),
)

**Interpretation.** The exact index check confirms that every saved prediction corresponds to the same purged validation feature date. In the saved experiment, 500 feature dates map to 502 close observations through 2021-12-28.

## 2. From predictions to positions

Prediction **1** targets a fully long SPY position. Prediction **0** targets cash. The signal uses information available after close *t*. If the target changes, the ledger trades at the next observed close *t+1*, using that close as a hypothetical fill price. For a new long position, the first subsequent price return runs from close *t+1* to close *t+2*, the interval used by the classifier’s label.

In [ ]:
model_positions = {}
for model in preds:
    signal = preds[model].rename('prediction')
    executions = map_signals_to_executions(calendar, signal)
    model_positions[model] = make_execution_positions(calendar, executions)

# The matched benchmark enters at the same first mapped close.
benchmark_signal = pd.Series(
    1, index=validation.index, dtype=int, name='always_long'
)
benchmark_executions = map_signals_to_executions(calendar, benchmark_signal)
benchmark_positions = make_execution_positions(calendar, benchmark_executions)
model_positions[BENCHMARK] = benchmark_positions

**Interpretation.** The models and the benchmark start in cash and share the same calendar. For the always-UP benchmark, the January 2, 2020 signal leads to a hypothetical purchase at the January 3 close. January 6 is the first subsequent valuation, so the first price return on those shares is measured from January 3 to January 6. Each target stays in effect until a later signal changes it.

## 3. Account for cash, shares and costs

With available cash $C$, execution close $P$, shares sold $q$ and proportional fee $f$, a purchase spends the available cash inclusive of fees:

$$
q_{\mathrm{buy}}=\frac{C}{P(1+f)}.
$$

A sale credits net proceeds:

$$
C_{\mathrm{sale}}=qP(1-f).
$$

Let $C_t$ and $q_t$ be closing cash and shares at observed close $P_t$. Closing portfolio equity is

$$
V_t=C_t+q_tP_t.
$$

At the same execution price, the fee $F_t$ accounts for the change in equity:

$$
V_t^{\mathrm{post}}=V_t^{\mathrm{pre}}-F_t.
$$

In [ ]:
curves = {}
summary_rows = []

for model, positions in model_positions.items():
    for scenario, bps in SCENARIOS.items():
        ledger = run_long_cash_ledger(
            calendar, positions, initial_cash=INITIAL_CASH, cost_bps=bps
        )
        curves[(model, scenario)] = ledger.equity
        summary_rows.append({
            'portfolio': model,
            'scenario': scenario,
            'ending_equity': ledger.equity.iloc[-1],
            'return': ledger.equity.iloc[-1] / INITIAL_CASH - 1,
            'max_drawdown': (
                ledger.equity / ledger.equity.cummax() - 1
            ).min(),
            'trades': int(ledger.execution_count.sum()),
        })

summary = pd.DataFrame(summary_rows)

## 4. Construct a matched benchmark

SPY buy-and-hold is built from an always-UP signal on the same validation feature dates. It enters on the same first execution date and is valued on the same close dates, with the same starting cash and each cost assumption. The logistic and training-majority prediction columns are all UP in this saved validation set, so their mapped positions, purchases, costs and marks must match the benchmark exactly. The assertions below preserve that reconciliation check.

In [ ]:
for model in ['Logistic regression', 'Training majority baseline']:
    for scenario in SCENARIOS:
        pd.testing.assert_series_equal(
            curves[(model, scenario)],
            curves[(BENCHMARK, scenario)],
            check_names=False,
        )

display(summary.assign(
    ending_equity=summary.ending_equity.map('${:,.2f}'.format),
    return_pct=summary['return'].map('{:.2%}'.format),
    max_drawdown=summary.max_drawdown.map('{:.2%}'.format),
).drop(columns='return').set_index(['portfolio', 'scenario']))

**Interpretation.** The assertions compare the logistic and training-majority equity paths with benchmark equity at each date under both cost assumptions. The positions match because all three signals are UP on every feature date and the same deterministic mapping assigns them to the same execution closes.

## 5. Compare hypothetical portfolios

Ending equity is the final marked account value; cumulative return is ending equity divided by starting capital, minus one. Maximum drawdown is the largest peak-to-subsequent-trough decline in the marked equity curve. Execution count sums the ledger’s purchase and sale trade legs. The plot and saved tables use the same model, benchmark and cost scenarios.

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, scenario in zip(axes, SCENARIOS):
    for (name, cost), curve in curves.items():
        if cost == scenario:
            ax.plot(
                curve.index,
                curve / INITIAL_CASH,
                label=name,
                linewidth=1.25,
            )
    ax.set_title(scenario.replace('_', ' '))
    ax.set_ylabel('Growth of $1')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=.25)
plt.tight_layout()

### Save equity curves, portfolio summaries and assumptions

In [ ]:
pd.DataFrame(curves).to_csv(
    OUT / 'portfolio_equity_curves.csv', index_label='date'
)
summary.to_csv(OUT / 'portfolio_summary.csv', index=False)
portfolio_manifest = {
    'input_predictions': 'validation_predictions.csv',
    'execution': 'next observed close after feature close',
    'valuation': 'following observed close',
    'initial_cash': INITIAL_CASH,
    'cost_scenarios_bps': list(SCENARIOS.values()),
    'benchmark': 'matched SPY buy-and-hold',
    'dividends_or_splits_credited': False,
    'reserved_test_access': False,
}
_ = (OUT / 'portfolio_provenance.json').write_text(
    json.dumps(portfolio_manifest, indent=2)
)

**Turnover and costs.** The MLP makes 246 trade legs in each scenario. Its ending equity falls from **USD 1,026.52** at zero cost to **USD 907.71** at 5 bps per leg. Maximum drawdown deepens from 23.98% to 24.24%. Fees depend on the amount traded and reduce the capital available for subsequent positions.

Ending equity marks remaining shares at the last observed close. The ledger assumes no terminal sale or exit fee.

## 6. A small accounting example

This four-session toy series is supplementary. It illustrates how the ledger records cash, shares, costs and equity; it does not generate or feed the real-data results above.

In [ ]:
toy_dates = pd.date_range('2024-01-02', periods=4, freq='B')
toy_prices = pd.Series(
    [100.0, 101.0, 102.0, 99.0], index=toy_dates, name='SPY'
)
toy_positions = pd.Series(
    [0, 1, 1, 0], index=toy_dates, dtype=int, name='target_position'
)
toy_ledger = run_long_cash_ledger(
    toy_prices, toy_positions, initial_cash=INITIAL_CASH, cost_bps=5
)
display(toy_ledger[[
    'target_position', 'signed_shares_traded', 'cash', 'shares', 'transaction_cost', 'equity'
]])

## Conclusion

A directional prediction determines a target position. Its contribution to equity depends on when that position is filled, the size of the subsequent price change and any fees. The matched buy-and-hold account gives those effects a common reference.

These hypothetical price-only accounts use previously examined validation dates, omit distributions and corporate actions, and assume zero cash yield. The costs are illustrative and fill availability is unverified. Investable total returns would require those assumptions to be addressed. Notebook 05 reconstructs the same portfolios with an independent order engine.